![NYC Skyline](nyc.jpg)

Welcome to New York City, one of the most-visited cities in the world. There are many Airbnb listings in New York City to meet the high demand for temporary lodging for travelers, which can be anywhere between a few nights to many months. In this project, we will take a closer look at the New York Airbnb market by combining data from multiple file types like `.csv`, `.tsv`, and `.xlsx`.

Recall that **CSV**, **TSV**, and **Excel** files are three common formats for storing data. 
Three files containing data on 2019 Airbnb listings are available to you:

**data/airbnb_price.csv**
This is a CSV file containing data on Airbnb listing prices and locations.
- **`listing_id`**: unique identifier of listing
- **`price`**: nightly listing price in USD
- **`nbhood_full`**: name of borough and neighborhood where listing is located

**data/airbnb_room_type.xlsx**
This is an Excel file containing data on Airbnb listing descriptions and room types.
- **`listing_id`**: unique identifier of listing
- **`description`**: listing description
- **`room_type`**: Airbnb has three types of rooms: shared rooms, private rooms, and entire homes/apartments

**data/airbnb_last_review.tsv**
This is a TSV file containing data on Airbnb host names and review dates.
- **`listing_id`**: unique identifier of listing
- **`host_name`**: name of listing host
- **`last_review`**: date when the listing was last reviewed

In [128]:
# Import necessary packages
import pandas as pd
import numpy as np

# Importing .csv file
airbnb_price = pd.read_csv('data/airbnb_price.csv')
print(airbnb_price.head())

   listing_id        price                nbhood_full
0        2595  225 dollars         Manhattan, Midtown
1        3831   89 dollars     Brooklyn, Clinton Hill
2        5099  200 dollars     Manhattan, Murray Hill
3        5178   79 dollars  Manhattan, Hell's Kitchen
4        5238  150 dollars       Manhattan, Chinatown


In [129]:
# Importing .tsv file with correct separator
airbnb_last_review = pd.read_csv('data/airbnb_last_review.tsv', sep='\t')
print(airbnb_last_review.head())

   listing_id    host_name   last_review
0        2595     Jennifer   May 21 2019
1        3831  LisaRoxanne  July 05 2019
2        5099        Chris  June 22 2019
3        5178     Shunichi  June 24 2019
4        5238          Ben  June 09 2019


In [130]:
# Importing .xlsx file
airbnb_room_type_xls = pd.ExcelFile('data/airbnb_room_type.xlsx')
print(airbnb_room_type_xls.sheet_names)

['airbnb_room_type']


In [131]:
# Parse the sheet into a DataFrame
airbnb_room_type = airbnb_room_type_xls.parse(0)
print(airbnb_room_type.head())

   listing_id                                description        room_type
0        2595                      Skylit Midtown Castle  Entire home/apt
1        3831            Cozy Entire Floor of Brownstone  Entire home/apt
2        5099  Large Cozy 1 BR Apartment In Midtown East  Entire home/apt
3        5178            Large Furnished Room Near B'way     private room
4        5238         Cute & Cozy Lower East Side 1 bdrm  Entire home/apt


In [132]:
# Combining the three DataFrames
airbnb = airbnb_room_type.merge(airbnb_last_review, on='listing_id', how='left') \
                            .merge(airbnb_price, on='listing_id', how='left')
print(airbnb.head())

   listing_id  ...                nbhood_full
0        2595  ...         Manhattan, Midtown
1        3831  ...     Brooklyn, Clinton Hill
2        5099  ...     Manhattan, Murray Hill
3        5178  ...  Manhattan, Hell's Kitchen
4        5238  ...       Manhattan, Chinatown

[5 rows x 7 columns]


In [133]:
print(airbnb.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25209 entries, 0 to 25208
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   listing_id   25209 non-null  int64 
 1   description  25199 non-null  object
 2   room_type    25209 non-null  object
 3   host_name    25201 non-null  object
 4   last_review  25209 non-null  object
 5   price        25209 non-null  object
 6   nbhood_full  25209 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.5+ MB
None


In [134]:
# Checking for null values
print(airbnb.isna().sum())

listing_id      0
description    10
room_type       0
host_name       8
last_review     0
price           0
nbhood_full     0
dtype: int64


In [135]:
# Filling in the missing values for the description and host_name
airbnb = airbnb.fillna({
    'description' : 'No description provided',
    'host_name' : 'Unknown'
})
print(airbnb.isna().sum())

listing_id     0
description    0
room_type      0
host_name      0
last_review    0
price          0
nbhood_full    0
dtype: int64


In [136]:
# Changing the data types
# Changing the price column from object to int
airbnb['price'] = airbnb['price'].str.replace(' dollars', '')
airbnb['price'] = airbnb['price'].astype('int')

# Changing the last_review column from object to datetime
airbnb['last_review'] = pd.to_datetime(airbnb['last_review'])

# Changing room_type to category
airbnb['room_type'].value_counts()

Entire home/apt    8458
Private room       7241
entire home/apt    2665
private room       2248
ENTIRE HOME/APT    2143
PRIVATE ROOM       1867
Shared room         380
shared room         110
SHARED ROOM          97
Name: room_type, dtype: int64

In [137]:
airbnb['room_type'] = airbnb['room_type'].str.strip()
airbnb['room_type'] = airbnb['room_type'].str.lower()
airbnb['room_type'].value_counts()
airbnb['room_type'] = airbnb['room_type'].astype('category')

In [138]:
# Changing remaining columns to strings
string_cols = airbnb.select_dtypes(include=['object']).columns
airbnb[string_cols] = airbnb[string_cols].astype('string')
airbnb.dtypes

listing_id              int64
description            string
room_type            category
host_name              string
last_review    datetime64[ns]
price                   int64
nbhood_full            string
dtype: object

In [139]:
# Finding the dates of the earliest and most recent reviews
earliest_review = airbnb['last_review'].min()
print(earliest_review)
recent_review = airbnb['last_review'].max()
print(recent_review)

2019-01-01 00:00:00
2019-07-09 00:00:00


In [140]:
# Finding the number of listings that are private rooms
private_rooms_count = len(airbnb[airbnb['room_type'] == 'private room'])
print(private_rooms_count)

11356


In [141]:
# Getting the average listing price
avg_listing_price = airbnb['price'].mean().round(2)
print(avg_listing_price)

141.78


In [142]:
# Combining variables
airbnb_dict = {
    'first_reviewed' : [earliest_review],
    'last_reviewed' : [recent_review],
    'nb_private_rooms' : [private_rooms_count],
    'avg_price' : [avg_listing_price]
}
review_dates = pd.DataFrame(airbnb_dict)
print(review_dates.head(1))

  first_reviewed last_reviewed  nb_private_rooms  avg_price
0     2019-01-01    2019-07-09             11356     141.78
